In [1]:

import pandas as pd
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

from sklearn.datasets import fetch_openml

data = fetch_openml(
    name="house_prices",
    version=1,
    as_frame=True
)

df = data.frame

print("Dataset shape:", df.shape)
display(df.head())
print("\nData types:")
print(df.dtypes.value_counts())


Dataset shape: (1460, 81)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000



Data types:
object     43
int64      35
float64     3
Name: count, dtype: int64


In [3]:

# Target: house sale price
target = "SalePrice"

X = df.drop(columns=[target])
y = df[target]

# Split first: test set must remain untouched during CV
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])
print("Features:", X_train.shape[1])


Training rows: 1168
Testing rows: 292
Features: 80


In [4]:

numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))


Numerical features: 37
Categorical features: 43


In [5]:

# Numerical preprocessing:
# 1. Fill missing values with median
# 2. Scale numerical features

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing:
# 1. Fill missing values with most frequent category
# 2. Convert categories into numeric columns

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Apply the correct preprocessing to each column type

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created successfully!")


Preprocessing pipeline created successfully!


In [6]:

model_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LinearRegression())
])

print(model_pipeline)


Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Id', 'MSSubClass',
                                                   'LotFrontage', 'LotArea',
                                                   'OverallQual', 'OverallCond',
                                                   'YearBuilt', 'YearRemodAdd',
                                                   'MasVnrArea', 'BsmtFinSF1',
                                                   'BsmtFinSF2', 'BsmtUnfSF',
                                                   'TotalBsmtSF', '1stFlrSF',
                

In [7]:

# Create 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Evaluate using training data only
cv_scores = cross_val_score(
    model_pipeline,
    X_train,
    y_train,
    cv=kf,
    scoring="r2"
)

print("R² score for each fold:")
for i, score in enumerate(cv_scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print("\nMean CV R²:", round(cv_scores.mean(), 4))
print("Standard deviation:", round(cv_scores.std(), 4))


R² score for each fold:
Fold 1: 0.8588
Fold 2: 0.7492
Fold 3: 0.6542
Fold 4: 0.8682
Fold 5: 0.7098

Mean CV R²: 0.7681
Standard deviation: 0.0836


In [8]:

# Fit the complete pipeline on the training data only
model_pipeline.fit(X_train, y_train)

# Predict on the untouched test set
y_pred = model_pipeline.predict(X_test)

# Evaluate final test performance
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Final Test Results")
print("------------------")
print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R² Score: {r2:.4f}")


Final Test Results
------------------
MAE:  $18,359.52
RMSE: $29,447.16
R² Score: 0.8869


## Cross-Validation and Leakage Prevention

The data was split into training and test sets before model evaluation. Five-fold cross-validation was performed using only the training set.

The preprocessing steps—including median imputation, standard scaling, and one-hot encoding—were included inside a single scikit-learn Pipeline. During each cross-validation fold, these transformations were fitted only on that fold's training portion.

This prevents information from the validation fold from influencing the preprocessing fitted on the training portion. The test set remained separate and was used only for final evaluation after cross-validation.

The mean cross-validation R² score was 0.7681, with a standard deviation of 0.0836. These scores indicate that performance varied across folds.

The final test metrics provide an independent evaluation of the fitted pipeline on data that was not used for training or cross-validation.


## Final Results and Conclusion

The complete machine-learning pipeline was evaluated using five-fold cross-validation and a separate test set.

The mean cross-validation R² score was 0.7681, with a standard deviation of 0.0836. On the held-out test set, the model achieved an R² score of 0.8869, an MAE of approximately $18,359.52, and an RMSE of approximately $29,447.16.

The pipeline combines numerical imputation and scaling with categorical imputation and one-hot encoding. Keeping these preprocessing steps inside the pipeline helps prevent data leakage because transformations are fitted only on the training portion of each fold.

The test set was kept separate from cross-validation and used for final evaluation only.

Overall, this project demonstrates how to build a preprocessing and regression pipeline, perform K-fold cross-validation, and evaluate the final model without leaking test data into training.

**Limitation:** The model's predictions have noticeable errors, as shown by MAE and RMSE. Further model comparison and error analysis could help improve performance.
